In [ ]:
# Step 1: match_sop(Incident: system failure during 2020-05-27 01:30:00 to 2020-05-27 02:00:00 (UTC+8). Root cause component, occurrence tim
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.50)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.44)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Log-Based Reason Analysis SOP (score 0.43)
  Name: RCA-Agent Log-Based Reason Analysis SOP
Steps:
1. get_logs(None, start_time, end_time): summarize log anomalies and raw error messages for candidate components in the incident window.
2. get_relevant_metric('error'): list error, resource, and traffic-related metric fields that may explain the logs.
3. whether_is_abnormal_metric(start_time, end_time, '', None): compare log evidence against metric anomalies and onset times.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 2: generate_sop_code for 'RCA-Agent Metric Fault Identification SOP'
start_time = '2020-05-27 01:30:00'
end_time = '2020-05-27 02:00:00'
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_obs = component_analyze(start_time, end_time)
answer = metric_evidence + '\n' + component_obs

```
Generated code for SOP 'RCA-Agent Metric Fault Identification SOP':
start_time = '2020-05-27 01:30:00'
end_time = '2020-05-27 02:00:00'
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_obs = component_analyze(start_time, end_time)
answer = metric_evidence + '\n' + component_obs
```

In [ ]:
# Step 3: run_sop
start_time = '2020-05-27 01:30:00'
end_time = '2020-05-27 02:00:00'
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_obs = component_analyze(start_time, end_time)
answer = metric_evidence + '\n' + component_obs

```
[ANOMALY] db_003 DbTime: rise to 192.8 at 2020-05-27 01:41:00 (median 11.9, thr 188.6, sev 27.5, n=2)
Component anomaly overview (ranked):
os_021  metric_anoms=9 (top: Disk_svctm sev=65.0)  trace_anoms=2  log_anoms=0
os_001  metric_anoms=9 (top: CPU_iowait_time sev=66.5)  trace_anoms=0  log_anoms=0
os_017  metric_anoms=9 (top: Disk_svctm sev=66.3)  trace_anoms=0  log_anoms=0
docker_003  metric_anoms=1 (top: container_cpu_used sev=35.6)  trace_anoms=8  log_anoms=0
docker_002  metric_anoms=1 (top: container_cpu_used sev=8.9)  trace_anoms=7  log_anoms=0
docker_004  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=8  log_anoms=0
db_003  metric_anoms=7 (top: Login_Per_Sec sev=63.9)  trace_anoms=0  log_anoms=0
docker_001  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=7  log_anoms=0
docker_005  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=7  log_anoms=0
docker_006  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=7  log_anoms=0
docker_008  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=7  log_anoms=0
os_018  metric_anoms=6 (top: Disk_svctm sev=52.8)  trace_anoms=0  log_anoms=0
os_020  metric_anoms=5 (top: Disk_wr_ios sev=35.8)  trace_anoms=0  log_anoms=0
os_009  metric_anoms=3 (top: Sent_queue sev=33.4)  trace_anoms=0  log_anoms=0
db_007  metric_anoms=2 (top: LFSync_Per_Sec sev=19.6)  trace_anoms=0  log_anoms=0
```

In [ ]:
# Step 4: match_observation([ANOMALY] db_003 DbTime: rise to 192.8 at 2020-05-27 01:41:00 (median 11.9, thr 188.6, sev 27.5, n=2)
Component anomaly 
...[truncated])

```
ObAgent: uncertain.

Reason: DbTime spike accompanied by wide OS anomalies in Disk_svctm and high CPU_iowait suggests an I/O/disk-induced DB slowdown, but no explicit disk fault class exists in the closed set. To disambiguate, check:
- whether_is_abnormal_metric(os_021, Disk_svctm) and same for os_001/os_017 (if true → IO/disk bottleneck causing DB latency).
- whether_is_abnormal_metric(db_003, Connections) and db_003,Connection_errors or Login_Per_Sec (to detect connection-limit).
- whether_is_abnormal_metric(network hosts, tcp_retransmits / rtt) (to detect network loss/delay).
Also collect a DB call-trace to see where time is spent (IO vs network vs queuing).
JudgeAgent: No - No — the run shows a db_003 DbTime spike at 2020-05-27 01:41:00 and widespread OS Disk_svctm/CPU_iowait anomalies, but the agent ended as 'uncertain' and did not pick a single component from the candidate list, did not map the issue to one of the closed-set types (CPU fault, network delay, network loss, db connection limit, db close), nor provide a concrete telemetry reason tied uniquely to one component. Root cause not fully pinned down.
```

In [ ]:
# Step 5: match_sop(Find SOPs for IO/disk-induced DB slowdown and trace-level DB call analysis. Evidence: db_003 DbTime spike to 192.8 at 20
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.40) (already executed)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.40)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Level and Root Selection SOP (score 0.35)
  Name: RCA-Agent Level and Root Selection SOP
Steps:
1. component_analyze(start_time, end_time): identify the strongest faulty candidates and their telemetry sources.
2. collect_trace(start_time, end_time): separate faulty upstream/downstream candidates when trace telemetry is available.
3. get_logs(None, start_time, end_time): check whether logs support the same component and reason as the metric/trace evidence.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 6: generate_sop_code for 'RCA-Agent Trace Localization SOP'
start_time = '2020-05-27 01:30:00'
end_time = '2020-05-27 02:00:00'
trace_evidence = collect_trace(start_time, end_time, None)
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_evidence + '\n' + metric_evidence

```
Generated code for SOP 'RCA-Agent Trace Localization SOP':
start_time = '2020-05-27 01:30:00'
end_time = '2020-05-27 02:00:00'
trace_evidence = collect_trace(start_time, end_time, None)
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_evidence + '\n' + metric_evidence
```

In [ ]:
# Step 7: run_sop
start_time = '2020-05-27 01:30:00'
end_time = '2020-05-27 02:00:00'
trace_evidence = collect_trace(start_time, end_time, None)
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_evidence + '\n' + metric_evidence

```
[TRACE] docker_004 trace.self.jdbc.duration_p95: rise to 0.0485 at 2020-05-27 01:38:00 (median 0.003, sev 56.1)
[TRACE] docker_007 trace.self.jdbc.duration_mean: rise to 0.005474 at 2020-05-27 01:47:00 (median 0.002534, sev 55.6)
[TRACE] docker_005 trace.self.local.duration_p95: rise to 0.023 at 2020-05-27 01:42:00 (median 0.009, sev 55.5)
[TRACE] docker_007 trace.self.local.duration_p95: rise to 0.024 at 2020-05-27 01:47:00 (median 0.01, sev 54.8)
[TRACE] docker_008 trace.self.local.duration_p95: rise to 0.023 at 2020-05-27 01:47:00 (median 0.01, sev 54.1)
[TRACE] docker_008 trace.self.jdbc.duration_mean: rise to 0.005289 at 2020-05-27 01:47:00 (median 0.002486, sev 47.3)
[TRACE] docker_005 trace.self.jdbc.duration_mean: rise to 0.005266 at 2020-05-27 01:47:00 (median 0.002439, sev 47.1)
[TRACE] docker_006 trace.self.jdbc.duration_mean: rise to 0.005138 at 2020-05-27 01:44:00 (median 0.00236, sev 44.6)
[TRACE] docker_001 trace.self.flyremote.duration_p95: rise to 0.06725 at 2020-05-27 01:47:00 (median 0.00705, sev 36.7)
[TRACE] docker_002 trace.self.flyremote.duration_p95: rise to 0.084 at 2020-05-27 01:33:00 (median 0.0095, sev 29.7)
[TRACE] docker_002 trace.to_docker_005.remoteprocess.duration_mean: rise to 0.03848 at 2020-05-27 01:41:00 (median 0.02504, sev 25.2)
[TRACE] docker_005 trace.from_docker_002.remoteprocess.duration_mean: rise to 0.03848 at 2020-05-27 01:41:00 (median 0.02504, sev 25.2)
[TRACE] docker_007 trace.self.local.duration_mean: rise to 0.01163 at 2020-05-27 01:47:00 (median 0.006559, sev 24.2)
[TRACE] docker_001 trace.to_docker_006.remoteprocess.duration_p95: rise to 0.063 at 2020-05-27 01:47:00 (median 0.03483, sev 23.0)
[TRACE] docker_006 trace.from_docker_001.remoteprocess.duration_p95: rise to 0.063 at 2020-05-27 01:47:00 (median 0.03483, sev 23.0)
Most-downstream faulty component(s): docker_001, docker_002, docker_003, docker_004, docker_005, docker_006, docker_007, docker_008, os_021
[ANOMALY] db_003 DbTime: rise to 192.8 at 2020-05-27 01:41:00 (median 11.9, thr 188.6, sev 27.5, n=2)
```

In [ ]:
# Step 8: match_observation([TRACE] docker_004 trace.self.jdbc.duration_p95: rise to 0.0485 at 2020-05-27 01:38:00 (median 0.003, sev 56.1)
[TRACE] 
...[truncated])

```
ObAgent: uncertain

Analysis: app JDBC and local trace latencies and remoteprocess durations rose across many containers while db_003 DbTime spiked — indicating the DB itself is slow or overloaded, not clearly a pure network latency. High Login_Per_Sec on db_003 and no reported connection errors could mean reconnect churn (db close) or connection queuing (connection limit); OS disk svctm/iowait anomalies point to possible DB host I/O issues. To disambiguate, check:
- whether_is_abnormal_metric for db_003: ActiveConnections, ConnectionsFailed/Refused, and Login_Failures;
- DB error logs for "too many connections" or abrupt connection closes;
- network metrics (tcp_retransmits, round-trip) between app and DB;
- DB host disk I/O (disk svctm, iowait).
JudgeAgent: No - Not found — agents remained 'uncertain' and did not pick a single component or map the issue to one of the closed-set types. Although there is a db_003 DbTime spike at 2020-05-27 01:41:00 and widespread OS Disk_svctm/CPU_iowait anomalies (os_021, os_001, os_017), no single location, closed-set anomaly type, nor a concrete telemetry reason uniquely tied to one component was confirmed.
```

In [ ]:
# Step 9: match_sop(Find SOPs for IO/disk-induced DB slowdown and DB connection issues. Evidence: db_003 DbTime spike to 192.8 at 2020-05-27
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.33) (already executed)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.31) (already executed)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Level and Root Selection SOP (score 0.26)
  Name: RCA-Agent Level and Root Selection SOP
Steps:
1. component_analyze(start_time, end_time): identify the strongest faulty candidates and their telemetry sources.
2. collect_trace(start_time, end_time): separate faulty upstream/downstream candidates when trace telemetry is available.
3. get_logs(None, start_time, end_time): check whether logs support the same component and reason as the metric/trace evidence.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 10: generate_sop_code for 'RCA-Agent Level and Root Selection SOP'
start_time = '2020-05-27 01:30:00'
end_time = '2020-05-27 02:00:00'

# Step 1: identify the strongest faulty candidates and their telemetry sources
overview = component_analyze(start_time, end_time)

# Step 2: separate faulty upstream/downstream candidates using trace telemetry
trace_evidence = collect_trace(start_time, end_time, None)

# Step 3: check logs to see if they support the same component and reason
logs = get_logs(None, start_time, end_time)

# Step 4: compile the observations from the former steps
answer = overview + '\n' + trace_evidence + '\n' + logs

```
Generated code for SOP 'RCA-Agent Level and Root Selection SOP':
start_time = '2020-05-27 01:30:00'
end_time = '2020-05-27 02:00:00'

# Step 1: identify the strongest faulty candidates and their telemetry sources
overview = component_analyze(start_time, end_time)

# Step 2: separate faulty upstream/downstream candidates using trace telemetry
trace_evidence = collect_trace(start_time, end_time, None)

# Step 3: check logs to see if they support the same component and reason
logs = get_logs(None, start_time, end_time)

# Step 4: compile the observations from the former steps
answer = overview + '\n' + trace_evidence + '\n' + logs
```

In [ ]:
# Step 11: run_sop
start_time = '2020-05-27 01:30:00'
end_time = '2020-05-27 02:00:00'

# Step 1: identify the strongest faulty candidates and their telemetry sources
overview = component_analyze(start_time, end_time)

# Step 2: separate faulty upstream/downstream candidates using trace telemetry
trace_evidence = collect_trace(start_time, end_time, None)

# Step 3: check logs to see if they support the same component and reason
logs = get_logs(None, start_time, end_time)

# Step 4: compile the observations from the former steps
answer = overview + '\n' + trace_evidence + '\n' + logs

```
Component anomaly overview (ranked):
os_021  metric_anoms=9 (top: Disk_svctm sev=65.0)  trace_anoms=2  log_anoms=0
os_001  metric_anoms=9 (top: CPU_iowait_time sev=66.5)  trace_anoms=0  log_anoms=0
os_017  metric_anoms=9 (top: Disk_svctm sev=66.3)  trace_anoms=0  log_anoms=0
docker_003  metric_anoms=1 (top: container_cpu_used sev=35.6)  trace_anoms=8  log_anoms=0
docker_002  metric_anoms=1 (top: container_cpu_used sev=8.9)  trace_anoms=7  log_anoms=0
docker_004  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=8  log_anoms=0
db_003  metric_anoms=7 (top: Login_Per_Sec sev=63.9)  trace_anoms=0  log_anoms=0
docker_001  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=7  log_anoms=0
docker_005  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=7  log_anoms=0
docker_006  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=7  log_anoms=0
docker_008  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=7  log_anoms=0
os_018  metric_anoms=6 (top: Disk_svctm sev=52.8)  trace_anoms=0  log_anoms=0
os_020  metric_anoms=5 (top: Disk_wr_ios sev=35.8)  trace_anoms=0  log_anoms=0
os_009  metric_anoms=3 (top: Sent_queue sev=33.4)  trace_anoms=0  log_anoms=0
db_007  metric_anoms=2 (top: LFSync_Per_Sec sev=19.6)  trace_anoms=0  log_anoms=0
[TRACE] docker_004 trace.self.jdbc.duration_p95: rise to 0.0485 at 2020-05-27 01:38:00 (median 0.003, sev 56.1)
[TRACE] docker_007 trace.self.jdbc.duration_mean: rise to 0.005474 at 2020-05-27 01:47:00 (median 0.002534, sev 55.6)
[TRACE] docker_005 trace.self.local.duration_p95: rise to 0.023 at 2020-05-27 01:42:00 (median 0.009, sev 55.5)
[TRACE] docker_007 trace.self.local.duration_p95: rise to 0.024 at 2020-05-27 01:47:00 (median 0.01, sev 54.8)
[TRACE] docker_008 trace.self.local.duration_p95: rise to 0.023 at 2020-05-27 01:47:00 (median 0.01, sev 54.1)
[TRACE] docker_008 trace.self.jdbc.duration_mean: rise to 0.005289 at 2020-05-27 01:47:00 (median 0.002486, sev 47.3)
[TRACE] docker_005 trace.self.jdbc.duration_mean: rise to 0.005266 at 2020-05-27 01:47:00 (median 0.002439, sev 47.1)
[TRACE] docker_006 trace.self.jdbc.duration_mean: rise to 0.005138 at 2020-05-27 01:44:00 (median 0.00236, sev 44.6)
[TRACE] docker_001 trace.self.flyremote.duration_p95: rise to 0.06725 at 2020-05-27 01:47:00 (median 0.00705, sev 36.7)
[TRACE] docker_002 trace.self.flyremote.duration_p95: rise to 0.084 at 2020-05-27 01:33:00 (median 0.0095, sev 29.7)
[TRACE] docker_002 trace.to_docker_005.remoteprocess.duration_mean: rise to 0.03848 at 2020-05-27 01:41:00 (median 0.02504, sev 25.2)
[TRACE] docker_005 trace.from_docker_002.remoteprocess.duration_mean: rise to 0.03848 at 2020-05-27 01:41:00 (median 0.02504, sev 25.2)
[TRACE] docker_007 trace.self.local.duration_mean: rise to 0.01163 at 2020-05-27 01:47:00 (median 0.006559, sev 24.2)
[TRACE] docker_001 trace.to_docker_006.remoteprocess.duration_p95: rise to 0.063 at 2020-05-27 01:47:00 (median 0.03483, sev 23.0)
[TRACE] docker_006 trace.from_docker_001.remoteprocess.duration_p95: rise to 0.063 at 2020-05-27 01:47:00 (median 0.03483, sev 23.0)
Most-downstream faulty component(s): docker_001, docker_002, docker_003, docker_004, docker_005, docker_006, docker_007, docker_008, os_021
Log telemetry is not available for this dataset. Rely on metrics (whether_is_abnormal_metric) and traces (collect_trace) instead.
```

In [ ]:
# Step 12: match_observation(Component anomaly overview (ranked):
os_021  metric_anoms=9 (top: Disk_svctm sev=65.0)  trace_anoms=2  log_anoms=0
os_00
...[truncated])

```
ObAgent: The fault class is likely to be CPU fault.

Reason: multiple OS nodes show strong CPU iowait rise (os_001 cpu_iowait sev=66.5) together with high disk service time (Disk_svctm on several hosts) and widespread JDBC/DB latency increases across containers and db_003 login/sec anomalies. This pattern is consistent with CPU waiting on I/O (iowait) causing backend request/DB latency. To further confirm, check host CPU iowait, overall CPU usage, and disk queue depth/IOPS with whether_is_abnormal_metric.
JudgeAgent: No - Agents remained uncertain. Although db_003 DbTime spiked at 2020-05-27 01:41:00 and several OS nodes (os_021, os_001, os_017) show Disk_svctm / CPU_iowait anomalies, the run did not select exactly one component from the candidate list, did not conclusively map the incident to one closed-set type, nor provide a single concrete telemetry reason tied uniquely to one component and start minute. ObAgent only hypothesized a CPU fault but did not confirm location or occurrence minute.
```